<a href="https://colab.research.google.com/github/zhafarulmaahiy/Pretrain_Yourtts/blob/main/Pretrain_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/idiap/coqui-ai-TTS.git

fatal: destination path 'coqui-ai-TTS' already exists and is not an empty directory.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

!pip install --upgrade pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install coqui-tts
!pip install tensorboard matplotlib ipython soundfile librosa
# Cek instalasi
import TTS
print(f"Coqui TTS version: {TTS.__version__}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Looking in indexes: https://download.pytorch.org/whl/cu121
Coqui TTS version: 0.27.5


In [3]:
import torch

print("=" * 50)
print("CUDA DEBUG INFO")
print("=" * 50)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")  # Harus: True
print(f"CUDA Device Count: {torch.cuda.device_count()}")  # Harus: >= 1
print(f"Current Device: {torch.cuda.current_device()}")
print(f"Device Name: {torch.cuda.get_device_name(0)}")

# Test tensor di GPU
try:
    test_tensor = torch.randn(100, 100).cuda()
    print("✅ Tensor dapat dipindah ke CUDA")
except:
    print("❌ ERROR: Tensor tidak bisa ke CUDA")
print("=" * 50)

CUDA DEBUG INFO
PyTorch Version: 2.10.0+cu128
CUDA Available: True
CUDA Device Count: 1
Current Device: 0
Device Name: NVIDIA L4
✅ Tensor dapat dipindah ke CUDA


In [4]:
import os
import torch
from pathlib import Path

BASE_PATH = "/content/drive/MyDrive/Pretraining"
DATASET_PATH = os.path.join(BASE_PATH, "vctk")
OUTPUT_PATH = os.path.join(BASE_PATH, "output")
# Buat folder output
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Cek struktur dataset
print("Struktur dataset:")
!ls -la "{DATASET_PATH}"

# Hitung jumlah speaker dan file audio
wav_path = os.path.join(DATASET_PATH, "wav48")
speakers = [d for d in os.listdir(wav_path) if os.path.isdir(os.path.join(wav_path, d))]
total_audio = sum([len(files) for r, d, files in os.walk(wav_path) if files])

print(f"\nJumlah speaker: {len(speakers)}")
print(f"Total file audio: {total_audio}")
print(f"Speaker list: {speakers[:5]}...")

Struktur dataset:
total 45651
-rw-------  1 root root     3548 Mar 14 12:11 config_se.json
-rw-------  1 root root  1094683 Mar 14 12:11 metadata.csv
-rw-------  1 root root  1027053 Mar 15 13:29 metadatas.csv
-rw-------  1 root root 44610930 Mar 14 12:11 model_se.pth.tar
-rw-------  1 root root      590 Mar 14 12:11 speaker-info.txt
drwx------ 43 root root     4096 Mar 15 12:08 txt
drwx------ 43 root root     4096 Mar 16 06:53 wav48

Jumlah speaker: 41
Total file audio: 13526
Speaker list: ['ID2JF09OR', 'ID2JF10OR', 'ID2JM08OR', 'ID2JM07OR', 'ID2JF14OR']...


In [5]:
# YourTTS memerlukan speaker encoder untuk mengekstrak d-vectors

# SPEAKER_ENCODER_URL = "https://github.com/coqui-ai/TTS/releases/download/speaker_encoder_model/model_se.pth.tar"
# SPEAKER_ENCODER_CONFIG_URL = "https://github.com/coqui-ai/TTS/releases/download/speaker_encoder_model/config_se.json"

# # Download ke folder output
# !wget -O "{OUTPUT_PATH}/model_se.pth.tar" {SPEAKER_ENCODER_URL}
# !wget -O "{OUTPUT_PATH}/config_se.json" {SPEAKER_ENCODER_CONFIG_URL}

# print("Speaker encoder model downloaded!")

In [6]:
import os
from glob import glob

DATASET_PATH = "/content/drive/MyDrive/Pretraining/vctk"

# Cek struktur direktori
print("=== Txt Files ===")
txt_files = glob(f"{os.path.join(DATASET_PATH, 'txt')}/**/*.txt", recursive=True)
print(f"Total txt files: {len(txt_files)}")
if txt_files:
    print(f"Sample: {txt_files[0]}")

print("\n=== Audio Files ===")
audio_files = glob(f"{os.path.join(DATASET_PATH, 'wav48')}/**/*.wav", recursive=True)
print(f"Total audio files: {len(audio_files)}")
if audio_files:
    print(f"Sample: {audio_files[0]}")

# Cek speaker IDs
print("\n=== Speaker IDs ===")
speaker_dirs = [d for d in os.listdir(os.path.join(DATASET_PATH, 'txt')) if os.path.isdir(os.path.join(DATASET_PATH, 'txt', d))]
print(f"Speaker IDs: {speaker_dirs[:5]}...")  # Show first 5

=== Txt Files ===
Total txt files: 13526
Sample: /content/drive/MyDrive/Pretraining/vctk/txt/ID2JF14OR/ID2JF14OR_0190.txt

=== Audio Files ===
Total audio files: 13526
Sample: /content/drive/MyDrive/Pretraining/vctk/wav48/ID2JF09OR/ID2JF09OR_0118.wav

=== Speaker IDs ===
Speaker IDs: ['ID2JF14OR', 'ID2LM03OR', 'ID2JF09OR', 'ID2JF10OR', 'ID2LF02OR']...


In [7]:
import os
from TTS.bin.compute_embeddings import compute_embeddings
from TTS.config.shared_configs import BaseDatasetConfig

BASE_PATH = "/content/drive/MyDrive/Pretraining"
DATASET_PATH = os.path.join(BASE_PATH, "vctk")
OUTPUT_PATH = os.path.join(BASE_PATH, "output")

embeddings_file = os.path.join(OUTPUT_PATH, "speakers2.pth")

dataset_config = BaseDatasetConfig(
    formatter="vctk_old",  # ✅ Ubah dari "vctk" ke "vctk_old"
    dataset_name="indonesian_dataset",
    path=DATASET_PATH,
    language="id",
    meta_file_train="",    # kosong string, bukan None
    meta_file_val="",
)

# compute_embeddings(
#     model_path="/content/drive/MyDrive/Pretraining/output/model_se.pth.tar",
#     config_path="/content/drive/MyDrive/Pretraining/output/config_se.json",
#     output_path=embeddings_file,
#     old_speakers_file=None,
#     old_append=False,
#     config_dataset_path=None,
#     formatter_name=dataset_config.formatter,      # ← "vctk_old"
#     dataset_name=dataset_config.dataset_name,
#     dataset_path=dataset_config.path,
#     meta_file_train=dataset_config.meta_file_train,
#     meta_file_val=dataset_config.meta_file_val,
#     disable_cuda=False,
#     no_eval=False,
# )

print("✅ Speaker embeddings computed successfully!")

✅ Speaker embeddings computed successfully!


In [8]:
import json
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.configs.vits_config import VitsConfig
from TTS.tts.models.vits import VitsArgs, VitsAudioConfig

# Audio configuration
SAMPLE_RATE = 16000  # Sesuaikan dengan sample rate dataset Anda
audio_config = VitsAudioConfig(
    sample_rate=SAMPLE_RATE,
    hop_length=256,
    win_length=1024,
    fft_size=1024,
    mel_fmin=0.0,
    mel_fmax=None,
    num_mels=80,
)

# Model arguments for YourTTS
model_args = VitsArgs(
    d_vector_file=[embeddings_file],  # File embeddings yang baru dibuat
    use_d_vector_file=True,
    d_vector_dim=512,
    num_layers_text_encoder=10,
    speaker_encoder_model_path=os.path.join(OUTPUT_PATH, "model_se.pth.tar"),
    speaker_encoder_config_path=os.path.join(OUTPUT_PATH, "config_se.json"),
    resblock_type_decoder="2",  # YourTTS menggunakan ResNet blocks type 2
    # Untuk bahasa Indonesia, kita mungkin perlu mengaktifkan ini nanti
    use_language_embedding=True,
    embedded_language_dim=4,
)

# Dataset configuration
dataset_config = BaseDatasetConfig(
    formatter="vctk_old",
    dataset_name="indonesian_dataset",
    path=DATASET_PATH,
    language="id",  # Bahasa Indonesia
    meta_file_train="",  # Bisa dikosongi, akan otomatis scan folder
    meta_file_val="",
)

# Karakter yang umum dalam teks Bahasa Indonesia
indonesian_chars = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
indonesian_chars += "àáâãäåæçèéêëìíîïðñòóôõöøùúûüýþÿ"
indonesian_chars += "ĀāĂăĄąĆćĈĉĊċČčĎďĐđĒēĔĕĖėĘęĚě"
indonesian_chars += "ĜĝĞğĠġĢģĤĥĦħĨĩĪīĬĭĮįİıĲĳĴĵĶķ"
indonesian_chars += "!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~ "

# Training configuration
config = VitsConfig(
    output_path=OUTPUT_PATH,
    model_args=model_args,
    run_name="YourTTS-Indonesian",
    project_name="YourTTS",
    run_description="Training YourTTS dengan dataset Bahasa Indonesia",
    dashboard_logger="tensorboard",
    logger_uri=None,
    audio=audio_config,

    # ✅ GPU SETTINGS
    cudnn_benchmark=True,  # ← PENTING: Enable cuDNN optimization
    cudnn_deterministic=False,  # False untuk performance (True untuk reproducibility)

    # BATCH & DATA LOADING
    batch_size=16,
    eval_batch_size=16,
    num_loader_workers=4,

    # TRAINING PARAMETERS
    eval_split_max_size=256,
    print_step=50,
    plot_step=100,
    log_model_step=500,
    save_step=200,
    save_n_checkpoints=3,
    save_checkpoints=True,
    target_loss="loss_1",
    print_eval=False,
    epochs=10,

    # MIXED PRECISION (untuk menghemat memory di GPU)
    mixed_precision=True,

    # TEXT & CHARACTER SETTINGS
    use_phonemes=False,
    text_cleaner="multilingual_cleaners",
    add_blank=True,

    # DATASET
    datasets=[dataset_config],

    # AUDIO LENGTH
    max_audio_len=SAMPLE_RATE * 10,  # Max 10 detik

    # TEST SENTENCES
    test_sentences=[
        ["Halo, selamat pagi. Ini adalah kalimat uji coba untuk model TTS Bahasa Indonesia.", None, None],
        ["Saya sedang belajar membuat model text to speech dengan YourTTS.", None, None],
        ["Bahasa Indonesia adalah bahasa yang indah dan kaya akan keragaman.", None, None],
    ],
)

# Simpan config ke file
config_path = os.path.join(OUTPUT_PATH, "config.json")
config.save_json(config_path)

print(f"Config file saved to: {config_path}")
!cat "{config_path}" | head -30

Config file saved to: /content/drive/MyDrive/Pretraining/output/config.json
{
    "output_path": "/content/drive/MyDrive/Pretraining/output",
    "logger_uri": null,
    "run_name": "YourTTS-Indonesian",
    "project_name": "YourTTS",
    "run_description": "Training YourTTS dengan dataset Bahasa Indonesia",
    "print_step": 50,
    "plot_step": 100,
    "model_param_stats": false,
    "wandb_entity": null,
    "dashboard_logger": "tensorboard",
    "save_on_interrupt": true,
    "log_model_step": 500,
    "save_step": 200,
    "save_n_checkpoints": 3,
    "save_checkpoints": true,
    "save_all_best": false,
    "save_best_after": 0,
    "target_loss": "loss_1",
    "print_eval": false,
    "test_delay_epochs": 0,
    "run_eval": true,
    "run_eval_steps": null,
    "distributed_backend": "nccl",
    "distributed_url": "tcp://localhost:54321",
    "mixed_precision": true,
    "precision": "fp16",
    "epochs": 10,
    "batch_size": 16,
    "eval_batch_size": 16,


In [9]:
from TTS.tts.datasets import load_tts_samples
from TTS.utils.audio import AudioProcessor

# Load audio processor
ap = AudioProcessor(**config.audio.to_dict())

# Load samples untuk verifikasi
train_samples, eval_samples = load_tts_samples(
    datasets=config.datasets,
    eval_split=True,
    eval_split_max_size=config.eval_split_max_size,
    eval_split_size=config.eval_split_size,
)

# print(f"Jumlah sample training: {len(train_samples)}")
# print(f"Jumlah sample evaluasi: {len(eval_samples)}")

# Tampilkan contoh sample
# if train_samples:
#     print("\nContoh sample:")
#     for i, sample in enumerate(train_samples[:3]):
#         print(f"Sample {i+1}:")
#         print(f"  Text: {sample['text'][:50]}...")
#         print(f"  Speaker: {sample['speaker_name']}")
#         print(f"  Audio: {sample['audio_file']}")

In [11]:
import os
import torch
from pathlib import Path

# ==================== KONFIGURASI ====================
BASE_PATH = "/content/drive/MyDrive/Pretraining"
OUTPUT_PATH = os.path.join(BASE_PATH, "output")
DATASET_PATH = os.path.join(BASE_PATH, "vctk")

# Path ke config.json dan embeddings
CONFIG_PATH = os.path.join(OUTPUT_PATH, "config.json")
EMBEDDINGS_FILE = os.path.join(OUTPUT_PATH, "speakers2.pth")  # ← Sesuaikan nama file

# =====================================================

from TTS.config import load_config
from TTS.tts.models.vits import Vits
from trainer import Trainer, TrainerArgs
from TTS.utils.audio import AudioProcessor
from TTS.tts.datasets import load_tts_samples

print("\n" + "="*70)
print("🚀 TRAINING DARI SCRATCH - CONFIG DARI JSON")
print("="*70 + "\n")

# --- STEP 1: Load config dari JSON ---
print("📋 Loading configuration from JSON...")
if not os.path.exists(CONFIG_PATH):
    print(f"❌ ERROR: Config file tidak ditemukan: {CONFIG_PATH}")
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH}")

config = load_config(CONFIG_PATH)
print("✅ Config loaded successfully from JSON")
print(f"   Run name: {config.run_name}")
print(f"   Model: {config.model}")
print(f"   Epochs: {config.epochs}")
print(f"   Batch size: {config.batch_size}")

# --- STEP 2: Update embeddings file path di config (jika diperlukan) ---
print("\n📁 Verifying embeddings file...")
if not os.path.exists(EMBEDDINGS_FILE):
    print(f"❌ ERROR: Embeddings file tidak ditemukan!")
    print(f"   Path: {EMBEDDINGS_FILE}")
    print(f"\n   File .pth yang ada di folder:")
    if os.path.exists(OUTPUT_PATH):
        for item in os.listdir(OUTPUT_PATH):
            if item.endswith(('.pth', '.pth.tar')):
                print(f"   - {item}")
    raise FileNotFoundError("Embeddings file tidak ditemukan!")

print(f"✅ Embeddings file found:")
print(f"   Path: {EMBEDDINGS_FILE}")
print(f"   Size: {os.path.getsize(EMBEDDINGS_FILE) / 1e9:.2f} GB")

# ✅ Update embeddings path di config (jika belum sesuai)
config.model_args.d_vector_file = [EMBEDDINGS_FILE]
config.model_args.use_d_vector_file = True
print(f"\n   ✅ Config updated with embeddings file")

# --- STEP 3: Verify dataset path ---
print("\n📊 Verifying dataset path...")
if not os.path.exists(DATASET_PATH):
    print(f"❌ ERROR: Dataset path tidak ditemukan: {DATASET_PATH}")
    raise FileNotFoundError(f"Dataset path not found: {DATASET_PATH}")

print(f"✅ Dataset path verified: {DATASET_PATH}")

# Check dataset structure
wav_path = os.path.join(DATASET_PATH, "wav48")
txt_path = os.path.join(DATASET_PATH, "txt")
print(f"   📁 wav48 folder: {'✅' if os.path.exists(wav_path) else '❌'}")
print(f"   📁 txt folder: {'✅' if os.path.exists(txt_path) else '❌'}")

# --- STEP 4: Load audio processor dari config ---
print("\n🎵 Loading audio processor from config...")
try:
    ap = AudioProcessor(**config.audio.to_dict())
    print("✅ Audio processor initialized")
    print(f"   Sample rate: {config.audio.sample_rate}")
    print(f"   Hop length: {config.audio.hop_length}")
except Exception as e:
    print(f"❌ ERROR loading audio processor: {e}")
    import traceback
    traceback.print_exc()
    raise

# --- STEP 5: Load training data ---
print("\n📚 Loading training samples...")
try:
    train_samples, eval_samples = load_tts_samples(
        datasets=config.datasets,
        eval_split=True,
        eval_split_max_size=config.eval_split_max_size,
        eval_split_size=config.eval_split_size,
    )
    print(f"✅ Training samples loaded:")
    print(f"   Train: {len(train_samples)} samples")
    print(f"   Eval: {len(eval_samples)} samples")

    # Show sample info
    if train_samples:
        sample = train_samples[0]
        print(f"\n   Sample info:")
        print(f"   - Text: {sample.get('text', 'N/A')[:50]}...")
        print(f"   - Speaker: {sample.get('speaker_name', 'N/A')}")
        print(f"   - Language: {sample.get('language', 'N/A')}")
except Exception as e:
    print(f"❌ ERROR loading samples: {e}")
    import traceback
    traceback.print_exc()
    raise

# --- STEP 6: Initialize model FROM SCRATCH ---
print("\n🤖 Initializing model from config...")
try:
    model = Vits.init_from_config(config)
    print("✅ Model initialized with RANDOM WEIGHTS (fresh start)")

    total_params = sum(p.numel() for p in model.parameters())
    print(f"   Total parameters: {total_params:,}")

    # Count trainable vs non-trainable
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Trainable parameters: {trainable_params:,}")

except Exception as e:
    print(f"❌ ERROR initializing model: {e}")
    import traceback
    traceback.print_exc()
    raise

# --- STEP 7: Setup trainer WITHOUT checkpoint (FRESH START) ---
print("\n🔧 Setting up trainer for FRESH TRAINING...")
print("   ⚠️  restore_path = None (NO checkpoint loaded)")
print("   ⚠️  Training will start from EPOCH 0")

try:
    # ✅ KUNCI: restore_path=None untuk fresh training
    trainer_args = TrainerArgs(
        restore_path=None,  # ← PENTING: None = training fresh
        skip_train_epoch=False,
        continue_path="",
    )
    print("✅ TrainerArgs configured for fresh training")
except Exception as e:
    print(f"❌ ERROR setting trainer args: {e}")
    import traceback
    traceback.print_exc()
    raise

# --- STEP 8: Create trainer ---
print("\n📍 Creating Trainer instance...")
try:
    trainer = Trainer(
        trainer_args,
        config,
        output_path=config.output_path,
        model=model,
        train_samples=train_samples,
        eval_samples=eval_samples,
        training_assets={"audio_processor": ap},
    )
    print("✅ Trainer created successfully")
except Exception as e:
    print(f"❌ ERROR creating trainer: {e}")
    import traceback
    traceback.print_exc()
    raise

# --- STEP 9: Display training configuration ---
print("\n" + "="*70)
print("📋 TRAINING CONFIGURATION SUMMARY")
print("="*70)
print(f"\n✅ Model Configuration:")
print(f"   - Model type: {config.model}")
print(f"   - Run name: {config.run_name}")
print(f"   - Output: {config.output_path}")

print(f"\n✅ Audio Configuration:")
print(f"   - Sample rate: {config.audio.sample_rate} Hz")
print(f"   - FFT size: {config.audio.fft_size}")
print(f"   - Hop length: {config.audio.hop_length}")
print(f"   - Number of mels: {config.audio.num_mels}")

print(f"\n✅ Multi-speaker Configuration:")
print(f"   - Use d-vectors: {config.model_args.use_d_vector_file}")
print(f"   - D-vector dimension: {config.model_args.d_vector_dim}")
print(f"   - Use language embedding: {config.model_args.use_language_embedding}")

print(f"\n✅ Training Configuration:")
print(f"   - Epochs: {config.epochs}")
print(f"   - Batch size: {config.batch_size}")
print(f"   - Eval batch size: {config.eval_batch_size}")
print(f"   - Save step: {config.save_step}")
print(f"   - Log step: {config.log_model_step}")
print(f"   - Mixed precision: {config.mixed_precision}")

print(f"\n✅ Hardware Configuration:")
print(f"   - cuDNN benchmark: {config.cudnn_benchmark}")
print(f"   - CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"   - GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"   - GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print(f"\n✅ Data Configuration:")
print(f"   - Training samples: {len(train_samples)}")
print(f"   - Eval samples: {len(eval_samples)}")
print(f"   - Eval split: {config.eval_split_size}")

# --- STEP 10: Verify CUDA ---
print("\n" + "="*70)
print("💻 HARDWARE INFORMATION")
print("="*70)
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"GPU Memory Reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")
    print(f"GPU Memory Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
else:
    print("⚠️  WARNING: CUDA tidak tersedia!")
    print("    Training akan menggunakan CPU - SANGAT LAMBAT!")

# --- STEP 11: Start FRESH training ---
print("\n" + "="*70)
print("🚀 STARTING FRESH TRAINING FROM SCRATCH")
print("="*70)
print(f"\n📊 Monitor dengan TensorBoard:")
print(f"   Command: !tensorboard --logdir {config.output_path}")
print(f"\n⏱️  Training duration:")
print(f"   - Total samples: {len(train_samples)}")
print(f"   - Epochs: {config.epochs}")
print(f"   - Steps per epoch: ~{len(train_samples) // config.batch_size}")
print(f"\n⚠️  CATATAN:")
print(f"   - Ini training FRESH (dari random initialization)")
print(f"   - Membutuhkan waktu lama (berjam-jam/berhari-hari)")
print(f"   - Checkpoint akan disimpan setiap {config.save_step} steps")
print(f"\n" + "="*70 + "\n")

try:
    print("⏱️  Training dimulai dari EPOCH 0...\n")
    trainer.fit()

    print("\n" + "="*70)
    print("✅ TRAINING SELESAI DENGAN SUKSES!")
    print("="*70)
    print(f"\n📁 Model dan checkpoint disimpan di:")
    print(f"   {config.output_path}")
    print(f"\n💾 Untuk melanjutkan training ke depan, gunakan program 'Resume Training'")

except KeyboardInterrupt:
    print("\n\n⚠️  TRAINING DIHENTIKAN OLEH USER (Ctrl+C)")
    print("="*70)
    print("💾 Checkpoint SUDAH DISIMPAN")
    print("\n   Untuk resume training, gunakan program 'Resume Training dari Checkpoint'")
    print("   dengan path ke checkpoint terakhir yang disimpan.\n")

except Exception as e:
    print(f"\n❌ ERROR SAAT TRAINING: {e}")
    print("="*70)
    import traceback
    traceback.print_exc()
    raise


🚀 TRAINING DARI SCRATCH - CONFIG DARI JSON

📋 Loading configuration from JSON...
✅ Config loaded successfully from JSON
   Run name: YourTTS-Indonesian
   Model: vits
   Epochs: 10
   Batch size: 16

📁 Verifying embeddings file...
✅ Embeddings file found:
   Path: /content/drive/MyDrive/Pretraining/output/speakers2.pth
   Size: 0.06 GB

   ✅ Config updated with embeddings file

📊 Verifying dataset path...
✅ Dataset path verified: /content/drive/MyDrive/Pretraining/vctk
   📁 wav48 folder: ✅
   📁 txt folder: ✅

🎵 Loading audio processor from config...
✅ Audio processor initialized
   Sample rate: 16000
   Hop length: 256

📚 Loading training samples...
✅ Training samples loaded:
   Train: 13391 samples
   Eval: 135 samples

   Sample info:
   - Text: keempat korban sempat dapat perawatan di rumah sak...
   - Speaker: VCTK_old_ID2LM01OR
   - Language: id

🤖 Initializing model from config...
✅ Model initialized with RANDOM WEIGHTS (fresh start)
   Total parameters: 86,797,248
   Trainable 

 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: True
 | > Precision: fp16
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 12
 | > Num. of Torch Threads: 6
 | > Torch seed: 54321
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: True
 | > Torch TF32 MatMul: False
 > Start Tensorboard: tensorboard --logdir=/content/drive/MyDrive/Pretraining/output/YourTTS-Indonesian-March-18-2026_11+25PM-0000000

 > Model has 86797248 parameters

 > EPOCH: 0/9
 --> /content/drive/MyDrive/Pretraining/output/YourTTS-Indonesian-March-18-2026_11+25PM-0000000


✅ Trainer created successfully

📋 TRAINING CONFIGURATION SUMMARY

✅ Model Configuration:
   - Model type: vits
   - Run name: YourTTS-Indonesian
   - Output: /content/drive/MyDrive/Pretraining/output

✅ Audio Configuration:
   - Sample rate: 16000 Hz
   - FFT size: 1024
   - Hop length: 256
   - Number of mels: 80

✅ Multi-speaker Configuration:
   - Use d-vectors: True
   - D-vector dimension: 512
   - Use language embedding: True

✅ Training Configuration:
   - Epochs: 10
   - Batch size: 16
   - Eval batch size: 16
   - Save step: 200
   - Log step: 500
   - Mixed precision: True

✅ Hardware Configuration:
   - cuDNN benchmark: True
   - CUDA available: True
   - GPU Device: NVIDIA L4
   - GPU Memory: 23.66 GB

✅ Data Configuration:
   - Training samples: 13391
   - Eval samples: 135
   - Eval split: 0.01

💻 HARDWARE INFORMATION
CUDA Available: True
GPU Device: NVIDIA L4
GPU Memory Total: 23.66 GB
GPU Memory Reserved: 0.36 GB
GPU Memory Allocated: 0.35 GB

🚀 STARTING FRESH TRAINING 


 > TRAINING (2026-03-18 23:28:00) 
 ! Run is kept in /content/drive/MyDrive/Pretraining/output/YourTTS-Indonesian-March-18-2026_11+25PM-0000000
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/trainer/trainer.py", line 1376, in fit
    self._fit()
  File "/usr/local/lib/python3.12/dist-packages/trainer/trainer.py", line 1328, in _fit
    self.train_epoch()
  File "/usr/local/lib/python3.12/dist-packages/trainer/trainer.py", line 1132, in train_epoch
    outputs, _ = self.train_step(batch, batch_num_steps, cur_step, loader_start_time)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/trainer/trainer.py", line 960, in train_step
    batch = self.format_batch(batch)
            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/trainer/trainer.py", line 688, in format_batch
    batch = self._get_model().format_batch(batch)
            ^^^^^^^^^^^^^^^

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/trainer/trainer.py", line 1376, in fit
    self._fit()
  File "/usr/local/lib/python3.12/dist-packages/trainer/trainer.py", line 1328, in _fit
    self.train_epoch()
  File "/usr/local/lib/python3.12/dist-packages/trainer/trainer.py", line 1132, in train_epoch
    outputs, _ = self.train_step(batch, batch_num_steps, cur_step, loader_start_time)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/trainer/trainer.py", line 960, in train_step
    batch = self.format_batch(batch)
            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/trainer/trainer.py", line 688, in format_batch
    batch = self._get_model().format_batch(batch)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/TTS/tts/models/vits.py", line 1233, in format_batch
    d_vectors = [d_vec

TypeError: object of type 'NoneType' has no len()

In [ ]:
from TTS.tts.models.vits import Vits
from trainer import Trainer, TrainerArgs

# Inisialisasi trainer
trainer_args = TrainerArgs(
    restore_path=None,  # Untuk fine-tuning, bisa diisi path checkpoint
    skip_train_epoch=False,
    continue_path="",  # Untuk resume training
)

# Inisialisasi model
model = Vits.init_from_config(config)

# Inisialisasi trainer
trainer = Trainer(
    trainer_args,
    config,
    output_path=config.output_path,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
    training_assets={"audio_processor": ap},
)

# Mulai training
print("Memulai training...")
print("Buka TensorBoard untuk memonitor progress:")
print(f"!tensorboard --logdir {os.path.join(OUTPUT_PATH, 'YourTTS-Indonesian')}")
print("\nTraining akan berjalan. Ini bisa memakan waktu berjam-jam hingga berhari-hari tergantung dataset.")

# Jalankan training
trainer.fit()

In [ ]:
# import os
# import glob
# import torch
# from pathlib import Path

# # ==================== KONFIGURASI ====================
# BASE_PATH = "/content/drive/MyDrive/Pretraining"
# OUTPUT_PATH = os.path.join(BASE_PATH, "output")
# DATASET_PATH = os.path.join(BASE_PATH, "vctk")

# # Apakah Anda ingin mencari checkpoint terakhir otomatis?
# AUTO_FIND_CHECKPOINT = True  # True = cari otomatis, False = manual path
# MANUAL_CHECKPOINT_PATH = ""  # Isi jika AUTO_FIND_CHECKPOINT = False

# # =====================================================

# from TTS.config import load_config
# from TTS.tts.models.vits import Vits
# from trainer import Trainer, TrainerArgs
# from TTS.utils.audio import AudioProcessor
# from TTS.tts.datasets import load_tts_samples

# print("\n" + "="*70)
# print("RESUME TRAINING DARI CHECKPOINT")
# print("="*70 + "\n")

# # --- STEP 1: Cari checkpoint terakhir ---
# if AUTO_FIND_CHECKPOINT:
#     print("🔍 Mencari checkpoint terakhir...")

#     # Cari semua checkpoint files
#     checkpoint_patterns = [
#         os.path.join(OUTPUT_PATH, "YourTTS-Indonesian-*", "checkpoint_*.pth.tar"),
#         os.path.join(OUTPUT_PATH, "YourTTS-Indonesian-*", "checkpoint_*.pth"),
#     ]

#     checkpoint_list = []
#     for pattern in checkpoint_patterns:
#         checkpoint_list.extend(glob.glob(pattern))

#     if not checkpoint_list:
#         print("❌ ERROR: Checkpoint tidak ditemukan!")
#         print(f"   Folder output yang ada:")
#         if os.path.exists(OUTPUT_PATH):
#             for item in os.listdir(OUTPUT_PATH):
#                 full_path = os.path.join(OUTPUT_PATH, item)
#                 if os.path.isdir(full_path):
#                     print(f"   📁 {item}/")
#                     try:
#                         for sub in os.listdir(full_path):
#                             print(f"      └─ {sub}")
#                     except:
#                         pass
#         raise FileNotFoundError("Tidak ada checkpoint ditemukan")

#     # Sort berdasarkan waktu modifikasi terbaru
#     checkpoint_list = sorted(checkpoint_list, key=os.path.getmtime, reverse=True)
#     CHECKPOINT_PATH = checkpoint_list[0]

#     print(f"✅ Checkpoint terakhir ditemukan:")
#     print(f"   Path: {CHECKPOINT_PATH}")
#     print(f"   Size: {os.path.getsize(CHECKPOINT_PATH) / 1e9:.2f} GB")
#     print(f"   Total checkpoint ditemukan: {len(checkpoint_list)}")
# else:
#     CHECKPOINT_PATH = MANUAL_CHECKPOINT_PATH
#     if not os.path.exists(CHECKPOINT_PATH):
#         print(f"❌ ERROR: Checkpoint path tidak ada: {CHECKPOINT_PATH}")
#         raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")
#     print(f"✅ Menggunakan checkpoint manual: {CHECKPOINT_PATH}")

# # --- STEP 2: Load config ---
# print("\n📋 Loading configuration...")
# CONFIG_PATH = os.path.join(OUTPUT_PATH, "config.json")
# if not os.path.exists(CONFIG_PATH):
#     print(f"❌ ERROR: Config tidak ditemukan: {CONFIG_PATH}")
#     raise FileNotFoundError(f"Config file not found: {CONFIG_PATH}")

# config = load_config(CONFIG_PATH)
# print("✅ Config loaded successfully")
# print(f"   Model: {config.model}")
# print(f"   Run name: {getattr(config, 'run_name', 'N/A')}")
# print(f"   Output path: {config.output_path}")

# # --- STEP 3: Load audio processor ---
# print("\n🎵 Loading audio processor...")
# ap = AudioProcessor(**config.audio.to_dict())
# print("✅ Audio processor ready")

# # --- STEP 4: Load training data ---
# print("\n📊 Loading training samples...")
# try:
#     train_samples, eval_samples = load_tts_samples(
#         datasets=config.datasets,
#         eval_split=True,
#         eval_split_max_size=getattr(config, 'eval_split_max_size', 256),
#         eval_split_size=getattr(config, 'eval_split_size', 0.05),
#     )
#     print(f"✅ Training samples loaded:")
#     print(f"   Train: {len(train_samples)} samples")
#     print(f"   Eval: {len(eval_samples)} samples")
# except Exception as e:
#     print(f"❌ ERROR loading samples: {e}")
#     import traceback
#     traceback.print_exc()
#     raise

# # --- STEP 5: Initialize model ---
# print("\n🤖 Initializing model...")
# try:
#     # ✅ BENAR - hanya parameter config yang diperlukan, tanpa verbose
#     model = Vits.init_from_config(config)
#     print("✅ Model initialized successfully")
#     print(f"   Total parameters: {sum(p.numel() for p in model.parameters()):,}")
# except Exception as e:
#     print(f"❌ ERROR initializing model: {e}")
#     import traceback
#     traceback.print_exc()
#     raise

# # --- STEP 6: Setup trainer dengan restore_path ---
# print("\n🔧 Setting up trainer with checkpoint...")
# try:
#     trainer_args = TrainerArgs(
#         restore_path=CHECKPOINT_PATH,  # Load checkpoint lama
#         skip_train_epoch=False,
#         continue_path="",
#     )
#     print("✅ TrainerArgs created")
# except Exception as e:
#     print(f"❌ ERROR creating TrainerArgs: {e}")
#     import traceback
#     traceback.print_exc()
#     raise

# # --- STEP 7: Create trainer ---
# print("📍 Creating Trainer...")
# try:
#     trainer = Trainer(
#         trainer_args,
#         config,
#         output_path=config.output_path,
#         model=model,
#         train_samples=train_samples,
#         eval_samples=eval_samples,
#         training_assets={"audio_processor": ap},
#     )
#     print("✅ Trainer created successfully")
# except Exception as e:
#     print(f"❌ ERROR creating Trainer: {e}")
#     import traceback
#     traceback.print_exc()
#     raise

# # --- STEP 8: Verify CUDA ---
# print("\n💻 Hardware Information:")
# print(f"   CUDA Available: {torch.cuda.is_available()}")
# if torch.cuda.is_available():
#     print(f"   GPU Device: {torch.cuda.get_device_name(0)}")
#     print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
#     print(f"   GPU Memory Reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")
#     print(f"   GPU Memory Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
# else:
#     print("   ⚠️  WARNING: CUDA tidak tersedia, training akan menggunakan CPU (SANGAT LAMBAT!)")

# # --- STEP 9: Start training ---
# print("\n" + "="*70)
# print("🚀 STARTING RESUME TRAINING")
# print("="*70)
# print(f"Checkpoint: {os.path.basename(CHECKPOINT_PATH)}")
# print(f"Output folder: {config.output_path}")
# print(f"Batch size: {config.batch_size}")
# print(f"Learning rate: {getattr(config, 'lr', 'default')}")
# print("="*70 + "\n")

# print("📊 Untuk monitor training, buka TensorBoard:")
# tensorboard_cmd = f"tensorboard --logdir {config.output_path}"
# print(f"   !{tensorboard_cmd}\n")

# try:
#     print("⏱️  Training dimulai...\n")
#     trainer.fit()

#     print("\n" + "="*70)
#     print("✅ TRAINING SELESAI DENGAN SUKSES")
#     print("="*70)
# except KeyboardInterrupt:
#     print("\n⚠️  Training dihentikan oleh user (Ctrl+C)")
# except Exception as e:
#     print(f"\n❌ ERROR saat training: {e}")
#     import traceback
#     traceback.print_exc()
#     raise